# Variation 2 — Kernel InfoNCE (Gaussian / Polynomial / Mixture)

Trains three `SparseHead` models, one per kernel function.  The Gaussian kernel
with σ²=0.07 is theoretically equivalent to standard InfoNCE (τ=0.07), serving
as a principled baseline.  The Mixture kernel has a learnable α.

In [ ]:
# [1] Config
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from src.config import CONFIG
print('cache_dir :', CONFIG['cache_dir'])
print('coco_root :', CONFIG['coco_root'])
print('results   :', CONFIG['results_dir'])
print('device    :', CONFIG['device'])

In [ ]:
# [2] Data — reuse cached CLIP embeddings if already computed by variation1
import clip, torch
from src.data_utils import cache_or_compute_embeddings, make_loader

device = CONFIG['device']
clip_model, preprocess = clip.load(CONFIG['clip_model'], device=device)
clip_model.eval()

train_img, train_txt = cache_or_compute_embeddings(clip_model, preprocess, 'train', CONFIG)
val_img,   val_txt   = cache_or_compute_embeddings(clip_model, preprocess, 'val',   CONFIG)
print(f'train: {train_img.shape}  val: {val_img.shape}')

train_loader = make_loader(train_img, train_txt, CONFIG['batch_size'])

In [ ]:
# [3] Model factory
from src.model import SparseHead

def make_head():
    return SparseHead(CONFIG['embed_dim'], CONFIG['sparse_dim']).to(device)

In [ ]:
# [4] Train one model per kernel
import torch.optim as optim
from functools import partial
from src.losses import gaussian_kernel, poly_kernel, MixtureKernel, kernel_infonce_loss
from src.train  import train_one_epoch, evaluate, save_checkpoint

# Mixture kernel is a Module so its α is included in the optimizer.
mixture_kernel = MixtureKernel(alpha_init=0.5).to(device)

kernels = {
    'gaussian': (gaussian_kernel, []),             # no extra params
    'poly':     (poly_kernel,     []),
    'mixture':  (mixture_kernel,  list(mixture_kernel.parameters())),
}

all_metrics = {}

for kernel_name, (kfn, extra_params) in kernels.items():
    print(f'\n─── Kernel: {kernel_name} ───')
    head = make_head()
    loss_fn = partial(kernel_infonce_loss, kernel_fn=kfn)

    opt = optim.AdamW(
        list(head.parameters()) + extra_params,
        lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay']
    )

    loss_curve = []
    for epoch in range(1, CONFIG['epochs'] + 1):
        l = train_one_epoch(head, train_loader, opt, loss_fn, device)
        loss_curve.append(l)
        extra = f'  α={mixture_kernel.alpha.item():.3f}' if kernel_name == 'mixture' else ''
        print(f'  epoch {epoch:3d}/{CONFIG["epochs"]}  loss={l:.4f}{extra}')

    metrics = evaluate(head, val_img, val_txt, CONFIG)
    metrics['loss_curve'] = loss_curve
    if kernel_name == 'mixture':
        metrics['final_alpha'] = mixture_kernel.alpha.item()
    all_metrics[kernel_name] = metrics
    print(metrics)
    save_checkpoint(head, metrics, kernel_name, 'variation2', CONFIG)

In [ ]:
# [5] Results table
k = CONFIG['retrieval_k']
print(f'\n{"Kernel":12s}  IR@{k}   TR@{k}   L0_img   Clarity  Cross-modal')
for name, m in all_metrics.items():
    print(f"{name:12s}  {m[f'IR@{k}']:.3f}   {m[f'TR@{k}']:.3f}   "
          f"{m['l0_img']:6.1f}   {m['clarity']:.4f}   {m['cross_modal']:.4f}")
if 'final_alpha' in all_metrics.get('mixture', {}):
    print(f"\nMixture final α = {all_metrics['mixture']['final_alpha']:.4f}")

In [ ]:
# [6] Comparison plot
import matplotlib.pyplot as plt

names = list(all_metrics.keys())
k = CONFIG['retrieval_k']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss curves
for name, m in all_metrics.items():
    axes[0].plot(m['loss_curve'], label=name)
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

# Retrieval@1
x = range(len(names))
ir = [all_metrics[n][f'IR@{k}'] for n in names]
tr = [all_metrics[n][f'TR@{k}'] for n in names]
axes[1].bar([i - 0.2 for i in x], ir, 0.4, label=f'IR@{k}')
axes[1].bar([i + 0.2 for i in x], tr, 0.4, label=f'TR@{k}')
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(names, rotation=10)
axes[1].set_title(f'Retrieval@{k}'); axes[1].legend()

# Clarity vs L0
cl = [all_metrics[n]['clarity'] for n in names]
l0 = [all_metrics[n]['l0_img']  for n in names]
ax2 = axes[2].twinx()
axes[2].bar([i - 0.2 for i in x], cl, 0.4, color='steelblue', label='Clarity')
ax2.bar(    [i + 0.2 for i in x], l0, 0.4, color='orange',    label='L0 img')
axes[2].set_xticks(list(x)); axes[2].set_xticklabels(names, rotation=10)
axes[2].set_ylabel('Clarity', color='steelblue')
ax2.set_ylabel('L0', color='orange')
axes[2].set_title('Clarity vs L0')
axes[2].legend(loc='upper left'); ax2.legend(loc='upper right')

fig.tight_layout()
out_path = CONFIG['results_dir'] / 'variation2' / 'comparison.png'
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Saved {out_path}')